# PCA-TVLDA-5: Fixed top configuration

This notebook removes search/CV and runs only the selected configuration from `PCA-TVLDA-4`. Toggle `RUN_ALL_SUBJECTS` to evaluate either Patient 1 only or all available subjects. Each row trains on that subject/stage TRAINING file and evaluates once on the matching TEST file.


In [1]:
from pathlib import Path
import csv

import numpy as np
import scipy.io as sio
from scipy.signal import butter, sosfiltfilt

DATA = Path("stroke-rehab")
RUN_ALL_SUBJECTS = True
SUBJECTS = ("P1", "P2", "P3") if RUN_ALL_SUBJECTS else ("P1",)
STAGES = ("pre", "post")
A, B = -1, 1  # A = right, B = left
EPS = 1e-12
RESULTS_CSV = "pca_tvlda_5_fixed_config_results.csv"

PREPROCESS_CONFIG = {
    "ref_mode": "car",
    "crop": (0, 6),
    "band": (13, 30),
    "win_s": 0.5,
    "step_s": 0.25,
}

# Stage-specific top model settings selected in PCA-TVLDA-4 for Patient 1.
MODEL_BY_STAGE = {
    "pre": {"k": 1, "pca_smooth": 2, "shrink": 0.0, "reg": 1e-6},
    "post": {"k": 4, "pca_smooth": 2, "shrink": 0.0, "reg": 1e-6},
}


In [2]:

#Practicamente mismo preprocesado que Elena
def load_trials(subject, stage, kind):
    m = sio.loadmat(DATA / f"{subject}_{stage}_{kind}.mat")
    fs = int(np.ravel(m["fs"])[0])
    eeg = m["y"]
    trig = m["trig"].ravel()
    X, y, i = [], [], 0
    while i < len(trig):
        if trig[i] not in (A, B):
            i += 1
            continue
        label, start = int(trig[i]), i
        while i < len(trig) and trig[i] == label:
            i += 1
        end = start + 8 * fs
        if i - start >= 8 * fs and end <= len(eeg):
            X.append(eeg[start:end].T)
            y.append(label)
    return np.asarray(X, float), np.asarray(y), fs

def assert_trials(y, name):
    vals, counts = np.unique(y, return_counts=True)
    assert len(y) == 80, f"{name}: expected 80 trials, got {len(y)}"
    assert dict(zip(vals, counts)) == {A: 40, B: 40}, f"{name}: label counts {dict(zip(vals, counts))}"

def car(X):
    return X - X.mean(axis=1, keepdims=True)

def bandpass(X, fs, band):
    sos = butter(4, band, btype="bandpass", fs=fs, output="sos")
    return sosfiltfilt(sos, X, axis=-1)

def features(X, fs, cfg=PREPROCESS_CONFIG):
    #CAR
    if cfg["ref_mode"] == "car":
        X = car(X)
    #BANDPASS FILTER
    X = bandpass(X, fs, cfg["band"])

    #WINDOWS
    cue = 2 * fs
    #INICIO DE TASK
    a = cue + round(cfg["crop"][0] * fs)
    #FINAL DE TASK
    b = cue + round(cfg["crop"][1] * fs)
    #WINDOW
    win = round(cfg["win_s"] * fs)
    #STEP SIZE (para el sliding window)
    step = round(cfg["step_s"] * fs)

    #Explicación: Aplicar varianza a cada window (np.var), luego aplicar logaritmo (np.log), 
    #Instead of raw EEG: channels x samples, the model sees: beta power over time per channel

    return np.stack([
        np.log(np.var(X[:, :, s:s + win], axis=-1) + EPS)
        for s in range(a, b - win + 1, step)
    ], axis=1)  # trials x time_windows x channels

def zscore_fit(X):
    mu = X.mean(axis=(0, 1), keepdims=True)
    sd = X.std(axis=(0, 1), keepdims=True) + EPS
    return mu, sd


In [3]:
def smooth_time(Z, radius):
    if radius <= 0:
        return Z
    return np.stack([Z[max(0, t - radius):t + radius + 1].mean(axis=0) for t in range(len(Z))])

def covs(X):
    return np.stack([np.atleast_2d(np.cov(X[:, t, :], rowvar=False)) for t in range(X.shape[1])])

class TVLDA:
    def __init__(self, reg=1e-6, shrink=0.0, smooth=0):
        self.reg = reg
        self.shrink = shrink
        self.smooth = smooth

    def fit(self, X, y):
        XA, XB = X[y == A], X[y == B]
        muA = smooth_time(XA.mean(axis=0), self.smooth)
        muB = smooth_time(XB.mean(axis=0), self.smooth)
        covA = smooth_time(covs(XA), self.smooth)
        covB = smooth_time(covs(XB), self.smooth)
        T, F = muA.shape
        self.w = np.empty((T, F))
        self.d = np.empty(T)
        for t in range(T):
            sw = 0.5 * (covA[t] + covB[t]) + self.reg * np.eye(F)
            if self.shrink > 0:
                sw = (1 - self.shrink) * sw + self.shrink * np.diag(np.diag(sw))
            self.w[t] = np.linalg.pinv(sw) @ (muB[t] - muA[t])
            self.d[t] = 0.5 * (muA[t] + muB[t]) @ self.w[t]
        return self

    def score(self, X):
        return (X * self.w[None]).sum(axis=(1, 2)) - self.d.sum()

    def predict(self, X):
        return np.where(self.score(X) >= 0, B, A)

def tvlda_pca_fit(X, y, model_cfg):
    W = TVLDA(reg=model_cfg["reg"], shrink=model_cfg["shrink"], smooth=model_cfg["pca_smooth"]).fit(X, y).w
    _, _, vt = np.linalg.svd(W, full_matrices=False)
    return vt[:min(model_cfg["k"], vt.shape[0])].T

def project(X, P):
    return np.einsum("ntf,fk->ntk", X, P)

def fit_predict(X_train, y_train, X_test, model_cfg):
    #Aplicar z-score normalization
    mu, sd = zscore_fit(X_train)
    X_train = (X_train - mu) / sd
    X_test = (X_test - mu) / sd
    # Crea un TVLDA aplicado a los datos, luego un PCA al resultado
    P = tvlda_pca_fit(X_train, y_train, model_cfg)
    #Aplicar TVLDA otra vez.l
    clf = TVLDA(reg=model_cfg["reg"], shrink=model_cfg["shrink"]).fit(project(X_train, P), y_train)
    return clf.predict(project(X_test, P))


In [4]:
def run_pair(subject, stage):
    #Load data
    Xtr_raw, ytr, fs = load_trials(subject, stage, "training")
    Xte_raw, yte, _ = load_trials(subject, stage, "test")
    # Double check que los datos sean correctos
    assert_trials(ytr, f"{subject}_{stage}_training")
    assert_trials(yte, f"{subject}_{stage}_test")
    #Aplicar CAR, Bandpass, y crear Windows
    Xtr = features(Xtr_raw, fs)
    Xte = features(Xte_raw, fs)
    # Las mejores configuraciones por cada etapa. Solo un dict con los parámetros
    model_cfg = MODEL_BY_STAGE[stage]
    # Aplicar z-score normalization, aplicar PCA, Aplicar TVLDA
    pred = fit_predict(Xtr, ytr, Xte, model_cfg)
    # Final Summary
    correct = int((pred == yte).sum())
    return {
        "subject": subject,
        "stage": stage,
        "accuracy": correct / len(yte),
        "correct": correct,
        "total": len(yte),
        "pre_config": f"ref=car band=13-30 crop=0-6s win=0.5 step=0.25 logvar",
        "model_config": f"k={model_cfg['k']} smooth={model_cfg['pca_smooth']} shrink={model_cfg['shrink']} reg={model_cfg['reg']}",
    }

rows = []
for subject in SUBJECTS:
    for stage in STAGES:
        row = run_pair(subject, stage)
        rows.append(row)
        print(f"{subject}_{stage:<4}  {row['accuracy']:6.2%}  ({row['correct']}/{row['total']})  {row['model_config']}")

with open(RESULTS_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

print(f"\nWrote {RESULTS_CSV}")


P1_pre   91.25%  (73/80)  k=1 smooth=2 shrink=0.0 reg=1e-06
P1_post  96.25%  (77/80)  k=4 smooth=2 shrink=0.0 reg=1e-06
P2_pre   76.25%  (61/80)  k=1 smooth=2 shrink=0.0 reg=1e-06
P2_post  95.00%  (76/80)  k=4 smooth=2 shrink=0.0 reg=1e-06
P3_pre   87.50%  (70/80)  k=1 smooth=2 shrink=0.0 reg=1e-06
P3_post  92.50%  (74/80)  k=4 smooth=2 shrink=0.0 reg=1e-06

Wrote pca_tvlda_5_fixed_config_results.csv
